In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df_crime_data = pd.read_csv('data/crimedata.csv')
sociodata = pd.read_csv('data/sociodata.csv')
population_data = pd.read_csv('data/populationdata.csv')

In [ ]:
sociodata

In [ ]:
population_area_and_pop = population_data[['GEOG', 'TOT_POP_2010']]
population_area_and_pop

In [ ]:
sociodata_and_population = sociodata.merge(population_area_and_pop, left_on='COMMUNITY AREA NAME', right_on='GEOG')

In [ ]:
sociodata_and_population.drop(columns=['GEOG'], inplace=True)

In [ ]:
sociodata_and_population

In [ ]:
mapping = {sociodata['Community Area Number'][i]: sociodata['COMMUNITY AREA NAME'][i] for i in range(len(sociodata))}

mapping

In [ ]:
crime_data = df_crime_data.copy()
crime_data['Community Area'] = crime_data['Community Area'].map(mapping)
crime_data['Community Area'].unique()


In [ ]:
crime_by_community = crime_data.groupby('Community Area').size().reset_index(name='Crime Count')
crime_by_community = crime_by_community.sort_values(by='Crime Count', ascending=False)
plt.figure(figsize=(6, 12))
plt.barh(crime_by_community['Community Area'][:10], crime_by_community['Crime Count'][:10])
plt.xlabel('Number of Crimes')
plt.title('Number of Crimes by Community Area')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
percentage_by_community = crime_by_community.copy()
total_crimes = percentage_by_community['Crime Count'].sum()
percentage_by_community['Crime Percentage'] = (percentage_by_community['Crime Count'] / total_crimes) * 100
percentage_by_community = percentage_by_community.sort_values(by='Crime Percentage', ascending=False)
plt.figure(figsize=(12, 6))
plt.bar(percentage_by_community['Community Area'][:10], percentage_by_community['Crime Percentage'][:10])
plt.xticks(rotation=90)
plt.xlabel('Community Area')
plt.ylabel('Percentage of Total Crimes')
plt.title('Percentage of Total Crimes by Community Area')
plt.tight_layout()
plt.show()

In [ ]:
sociodata

In [ ]:
aged_without_diplome = sociodata['PERCENT AGED 25+ WITHOUT HIGH SCHOOL DIPLOMA'][:-1]
sorted_indices = np.argsort(aged_without_diplome)
aged_without_diplome = aged_without_diplome[sorted_indices]
sociodata['COMMUNITY AREA NAME'] = sociodata['COMMUNITY AREA NAME'][:-1]
sociodata['COMMUNITY AREA NAME'] = sociodata['COMMUNITY AREA NAME'][sorted_indices]

plt.figure(figsize=(12, 6))
plt.bar(sociodata['COMMUNITY AREA NAME'][:10][::-1], aged_without_diplome[:10][::-1])
plt.xticks(rotation=90)
plt.xlabel('Community Area')
plt.ylabel('Top 10 %Population Aged 25+ Without High School Diploma')
plt.title('Top 10 %Population Aged 25+ Without High School Diploma by Community Area')
plt.tight_layout()
plt.show()

In [ ]:
if crime_by_community['Community Area'].iloc[0] not in sociodata['COMMUNITY AREA NAME'].values:
    print(f"Community Area '{crime_by_community['Community Area'].iloc[0]}' is not present in the sociological data.")

In [ ]:
from sklearn.linear_model import LinearRegression
merged_data = pd.merge(
    crime_by_community,
    sociodata,
    left_on='Community Area',
    right_on='COMMUNITY AREA NAME',
    how='inner'
)

merged_data = merged_data[merged_data['Community Area'] != 'CHICAGO']

feature_cols = ["HARDSHIP INDEX"]
# feature_cols = [
#     col for col in sociodata.columns
#     if col not in ['Community Area Number', 'COMMUNITY AREA NAME']
# ]

ncols = 3
nrows = int(np.ceil(len(feature_cols) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5 * nrows))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    x = merged_data[col].values.reshape(-1, 1)
    y = merged_data['Crime Count'].values

    model = LinearRegression().fit(x, y)
    y_pred = model.predict(x)
    r2 = model.score(x, y)

    axes[i].scatter(merged_data[col], merged_data['Crime Count'], alpha=0.7)
    x_sorted = np.sort(merged_data[col].values)
    axes[i].plot(x_sorted, model.predict(x_sorted.reshape(-1, 1)), color='red')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Crime Count')
    axes[i].set_title(f'Crime Count vs {col}\nR² = {r2:.3f}')

for j in range(len(feature_cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.grid()
plt.legend()
# plt.savefig('../docs/figures/crime_vs_hardship.png')
# plt.show()

plt.figure(figsize=(12, 6), dpi=500)
plt.scatter(merged_data['HARDSHIP INDEX'], merged_data['Crime Count'], alpha=0.7)
plt.plot(merged_data['HARDSHIP INDEX'], model.predict(merged_data['HARDSHIP INDEX'].values.reshape(-1, 1)), color='red')
plt.xlabel('Hardship Index')
plt.ylabel('Crime Count')
plt.title('Crime Count vs Hardship Index (Community Area)\nR² = {:.3f}'.format(r2))
plt.grid()
plt.tight_layout()
plt.savefig('../docs/figures/crime_vs_hardship.png')
plt.show()


In [ ]:
# > * For each commuinity in your dataset, compute the **conditional crime profile**: for each of, calculate
# >
# >   $$r(\text{crime}, \text{district}) = \frac{P(\text{crime} \mid \text{district})}{P(\text{crime})}$$

conditional_crime_profile = merged_data.copy()
total_crimes = conditional_crime_profile['Crime Count'].sum()
conditional_crime_profile['P(crime)'] = total_crimes / total_crimes
conditional_crime_profile['P(crime | district)'] = conditional_crime_profile['Crime Count'] / total_crimes
conditional_crime_profile['r(crime, district)'] = conditional_crime_profile['P(crime | district)'] / conditional_crime_profile['P(crime)']

conditional_crime_profile[['Community Area', 'r(crime, district)']].sort_values(by='r(crime, district)')

# create a bar plot of the conditional crime profile
plt.figure(figsize=(12, 6))
plt.bar(conditional_crime_profile['Community Area'], conditional_crime_profile['r(crime, district)'])
plt.xticks(rotation=90)
plt.xlabel('Community Area')
plt.ylabel('r(crime, district)')
plt.title('Conditional Crime Profile by Community Area')
plt.tight_layout()
plt.show()

In [ ]:
# create a plot of the crimes in each community area to see if it is normally distributed
plt.figure(figsize=(12, 6))
plt.hist(merged_data['Crime Count'], bins=20, edgecolor='black')
plt.xlabel('Crime Count')
plt.ylabel('Frequency')
plt.title('Distribution of Crime Counts by Community Area')
plt.tight_layout()
plt.show()

In [ ]:
crime_types_count = crime_data['Primary Type'].value_counts().reset_index()
crime_types_count.columns = ['Primary Type', 'Count']
# make the bar plot vertical such that labels are on the y-axis and the bars are horizontal
plt.figure(figsize=(12, 6))
plt.barh(crime_types_count['Primary Type'], crime_types_count['Count'])
plt.xlabel('Number of Crimes')
plt.title('Top 10 Crime Types')
plt.tight_layout()
plt.show()

In [ ]:
theft_categories = ["THEFT"]
theft_data = crime_data[crime_data['Primary Type'].isin(theft_categories)].copy()

crime_by_community = theft_data.groupby('Community Area').size().reset_index(name='Crime Count')
crime_by_community = crime_by_community.sort_values(by='Crime Count', ascending=False)

merged_data = pd.merge(
    crime_by_community,
    sociodata,
    left_on='Community Area',
    right_on='COMMUNITY AREA NAME',
    how='inner'
)

merged_data = merged_data[merged_data['Community Area'] != 'CHICAGO']

feature_cols = ["PERCENT AGED UNDER 18 OR OVER 64", "PER CAPITA INCOME "]


# feature_cols = [
#     col for col in sociodata.columns
#     if col not in ['Community Area Number', 'COMMUNITY AREA NAME']
# ]

ncols = 1
nrows = int(np.ceil(len(feature_cols) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5 * nrows), dpi=500)
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    x = merged_data[col].values.reshape(-1, 1)
    y = merged_data['Crime Count'].values

    model = LinearRegression().fit(x, y)
    y_pred = model.predict(x)
    r2 = model.score(x, y)

    axes[i].scatter(merged_data[col], merged_data['Crime Count'], alpha=0.7)
    x_sorted = np.sort(merged_data[col].values)
    axes[i].plot(x_sorted, model.predict(x_sorted.reshape(-1, 1)), color='red')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('THEFT Count')
    axes[i].set_title(f'THEFT Count vs {col} (Community Area)\nR² = {r2:.3f}')
    axes[i].grid()

for j in range(len(feature_cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.savefig('../docs/figures/theft_vs_socioeconomic.png')
plt.show()


In [ ]:
theft_data = crime_data[crime_data['Primary Type'] == 'NARCOTICS'].copy()

crime_by_community = theft_data.groupby('Community Area').size().reset_index(name='Crime Count')
crime_by_community = crime_by_community.sort_values(by='Crime Count', ascending=False)

merged_data = pd.merge(
    crime_by_community,
    sociodata,
    left_on='Community Area',
    right_on='COMMUNITY AREA NAME',
    how='inner'
)

merged_data = merged_data[merged_data['Community Area'] != 'CHICAGO']

feature_cols = [
    col for col in sociodata.columns
    if col not in ['Community Area Number', 'COMMUNITY AREA NAME']
]

ncols = 3
nrows = int(np.ceil(len(feature_cols) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5 * nrows))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    x = merged_data[col].values.reshape(-1, 1)
    y = merged_data['Crime Count'].values

    model = LinearRegression().fit(x, y)
    y_pred = model.predict(x)
    r2 = model.score(x, y)

    axes[i].scatter(merged_data[col], merged_data['Crime Count'], alpha=0.7)
    x_sorted = np.sort(merged_data[col].values)
    axes[i].plot(x_sorted, model.predict(x_sorted.reshape(-1, 1)), color='red')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('GAMBLING Count')
    axes[i].set_title(f'GAMBLING Count vs {col}\nR² = {r2:.3f}')

for j in range(len(feature_cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


In [ ]:
theft_data = crime_data[crime_data['Primary Type'] == 'BATTERY'].copy()

crime_by_community = theft_data.groupby('Community Area').size().reset_index(name='Crime Count')
crime_by_community = crime_by_community.sort_values(by='Crime Count', ascending=False)

merged_data = pd.merge(
    crime_by_community,
    sociodata,
    left_on='Community Area',
    right_on='COMMUNITY AREA NAME',
    how='inner'
)

merged_data = merged_data[merged_data['Community Area'] != 'CHICAGO']

feature_cols = [
    col for col in sociodata.columns
    if col not in ['Community Area Number', 'COMMUNITY AREA NAME']
]

ncols = 3
nrows = int(np.ceil(len(feature_cols) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5 * nrows))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    x = merged_data[col].values.reshape(-1, 1)
    y = merged_data['Crime Count'].values

    model = LinearRegression().fit(x, y)
    y_pred = model.predict(x)
    r2 = model.score(x, y)

    axes[i].scatter(merged_data[col], merged_data['Crime Count'], alpha=0.7)
    x_sorted = np.sort(merged_data[col].values)
    axes[i].plot(x_sorted, model.predict(x_sorted.reshape(-1, 1)), color='red')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('BATTERY Count')
    axes[i].set_title(f'BATTERY Count vs {col}\nR² = {r2:.3f}')

for j in range(len(feature_cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


In [ ]:
theft_data = crime_data[crime_data['Primary Type'] == 'GAMBLING'].copy()

crime_by_community = theft_data.groupby('Community Area').size().reset_index(name='Crime Count')
crime_by_community = crime_by_community.sort_values(by='Crime Count', ascending=False)

merged_data = pd.merge(
    crime_by_community,
    sociodata,
    left_on='Community Area',
    right_on='COMMUNITY AREA NAME',
    how='inner'
)

merged_data = merged_data[merged_data['Community Area'] != 'CHICAGO']

feature_cols = [
    col for col in sociodata.columns
    if col not in ['Community Area Number', 'COMMUNITY AREA NAME']
]

ncols = 3
nrows = int(np.ceil(len(feature_cols) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5 * nrows))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    x = merged_data[col].values.reshape(-1, 1)
    y = merged_data['Crime Count'].values

    model = LinearRegression().fit(x, y)
    y_pred = model.predict(x)
    r2 = model.score(x, y)

    axes[i].scatter(merged_data[col], merged_data['Crime Count'], alpha=0.7)
    x_sorted = np.sort(merged_data[col].values)
    axes[i].plot(x_sorted, model.predict(x_sorted.reshape(-1, 1)), color='red')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('NARCOTICS Count')
    axes[i].set_title(f'NARCOTICS Count vs {col}\nR² = {r2:.3f}')

for j in range(len(feature_cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()
